## LG Aimers 9기 — R/F 라우팅 전체 학습 및 세 번째 제출 (Colab)

07b에서 사전 선언한 1순위 경로를 전체 데이터로 재현한다. LightGBM과 65:35 logit 비율은 유지하고, R 행은 기존 CatBoost, F 행은 `all R + 2024 F`로 학습한 CatBoost를 사용한다. test 통계나 행 간 집계는 사용하지 않는다.

### 0. Drive 마운트와 프로젝트 확인

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile

if Path('/content').exists():
    from google.colab import drive
    drive.mount('/content/drive')

candidates = [
    Path('/content/drive/MyDrive/LG_aimers_9th'),
    Path('/content/drive/MyDrive/LG_Aimers_9th'),
    Path.cwd(),
    Path.cwd().parent,
]
required_relative = [
    Path('data/train.csv'),
    Path('data/test.csv'),
    Path('data/sample_submission.csv'),
    Path('src/features.py'),
    Path('src/submission_entrypoint.py'),
    Path('experiments/04_train_submit.py'),
    Path('experiments/07c_train_submit.py'),
    Path('outputs/f_routing_v1/latest_run.json'),
]
PROJECT_ROOT = next(
    (path for path in candidates if all((path / rel).exists() for rel in required_relative)),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('프로젝트 폴더 또는 07b 결과가 없습니다. required_relative을 확인하세요.')
PROJECT_ROOT = PROJECT_ROOT.resolve()
print('PROJECT_ROOT:', PROJECT_ROOT)
print('train.csv size (GB):', round((PROJECT_ROOT / 'data/train.csv').stat().st_size / 1e9, 3))

### 1. 패키지와 GPU 확인

A100은 두 CatBoost를 순차 학습하는 데 사용한다. LightGBM은 같은 설정으로 CPU 학습한다. NumPy와 pandas는 Colab 기본 버전을 유지한다.

In [ ]:
install = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm==4.6.0', 'catboost==1.2.8'],
    capture_output=True, text=True,
)
if install.returncode != 0:
    print(install.stdout)
    print(install.stderr)
    raise RuntimeError('모델 패키지 설치 실패')

import catboost
import lightgbm
import numpy as np
import pandas as pd

gpu_binary = shutil.which('nvidia-smi')
gpu_check = subprocess.run(
    [gpu_binary, '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True,
) if gpu_binary else None
gpu_available = gpu_check is not None and gpu_check.returncode == 0
print('Python:', sys.version.split()[0])
print('LightGBM:', lightgbm.__version__, '| CatBoost:', catboost.__version__)
print('GPU:', gpu_check.stdout.strip() if gpu_available else 'not detected (CPU fallback)')

### 2. 07b 선택 결과 재확인

임의로 새 경로를 고르지 않는다. 실행 완료된 07b의 서명, 1순위 통과 상태, 전체 연도 안전 게이트를 먼저 확인한다.

In [ ]:
ROUTING_LATEST = PROJECT_ROOT / 'outputs/f_routing_v1/latest_run.json'
with ROUTING_LATEST.open(encoding='utf-8') as handle:
    routing_latest = json.load(handle)
routing_summary = routing_latest['summary']
recorded_run = Path(routing_latest['run_dir'])
run_candidates = [recorded_run, ROUTING_LATEST.parent / recorded_run.name]
ROUTING_RUN = next((path for path in run_candidates if path.exists()), None)
if ROUTING_RUN is None:
    raise FileNotFoundError(f'07b run 폴더 없음: {run_candidates}')
route_table = pd.read_csv(ROUTING_RUN / 'route_summary.csv')
selected = route_table[route_table['route'].eq('current_r__latest_f1_f')]
assert routing_summary['signature'] == '46f3d7d0000e'
assert routing_summary['selected_route'] == 'current_r__latest_f1_f'
assert routing_summary['selection_status'] == 'primary_passed'
assert routing_summary['selected_passes_fixed_gate'] is True
assert len(selected) == 1 and bool(selected.iloc[0]['passes_fixed_gate'])
display(selected[[
    'route', 'recent_brier', 'recent_bss_raw',
    'delta_recent_brier_vs_base', 'delta_recent_f_brier_vs_base',
    'delta_brier_2022_vs_base', 'delta_brier_2023_vs_base',
    'delta_brier_2024_vs_base', 'passes_fixed_gate',
]])

### 3. 최종 학습 설정

In [ ]:
THREADS = max(1, min(8, os.cpu_count() or 2))
CATBOOST_TASK_TYPE = 'GPU' if gpu_available else 'CPU'
OUTPUT_DIR = PROJECT_ROOT / 'outputs/final_submission_v3'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print({
    'threads': THREADS,
    'catboost_task_type': CATBOOST_TASK_TYPE,
    'lgbm_estimators': 500,
    'catboost_iterations_each': 85,
    'route': 'R=base CatBoost, F=all R + 2024 F CatBoost',
    'output_dir': str(OUTPUT_DIR),
})

### 4. 입력 데이터와 라우팅 행 사전 점검

In [ ]:
train_check = pd.read_csv(
    PROJECT_ROOT / 'data/train.csv',
    usecols=['row_id', 'season', 'game_type', 'control_success'],
    encoding='utf-8-sig',
)
test_check = pd.read_csv(
    PROJECT_ROOT / 'data/test.csv',
    usecols=['row_id', 'game_type'], encoding='utf-8-sig',
)
sample_check = pd.read_csv(
    PROJECT_ROOT / 'data/sample_submission.csv',
    usecols=['row_id'], encoding='utf-8-sig',
)
route_mask = train_check['game_type'].eq('R') | (
    train_check['game_type'].eq('F') & train_check['season'].eq(2024)
)
assert sorted(train_check['season'].unique().tolist()) == [2019, 2020, 2021, 2022, 2023, 2024]
assert set(train_check['game_type'].unique()) <= {'R', 'F'}
assert set(test_check['game_type'].unique()) <= {'R', 'F'}
assert test_check['row_id'].astype('string').equals(sample_check['row_id'].astype('string'))
display(pd.DataFrame([
    {'frame': 'all/base CatBoost', 'rows': len(train_check), 'target_rate': train_check['control_success'].mean(), 'F_rows': train_check['game_type'].eq('F').sum()},
    {'frame': 'F-route CatBoost', 'rows': int(route_mask.sum()), 'target_rate': train_check.loc[route_mask, 'control_success'].mean(), 'F_rows': int((route_mask & train_check['game_type'].eq('F')).sum())},
]))
print('test game_type:', test_check['game_type'].value_counts(dropna=False).to_dict())
del train_check, test_check, sample_check, route_mask

### 5. 전체 학습 및 `submit_03.zip` 생성

CatBoost 두 개를 순차 학습하므로 05c보다 시간이 더 걸린다. 셀 마지막의 `All checks passed.`가 나오기 전에는 제출하지 않는다.

In [ ]:
cmd = [
    sys.executable, str(PROJECT_ROOT / 'experiments/07c_train_submit.py'),
    '--train-path', str(PROJECT_ROOT / 'data/train.csv'),
    '--data-dir', str(PROJECT_ROOT / 'data'),
    '--routing-latest-path', str(ROUTING_LATEST),
    '--output-dir', str(OUTPUT_DIR),
    '--threads', str(THREADS),
    '--lgbm-estimators', '500',
    '--catboost-iterations', '85',
    '--catboost-task-type', CATBOOST_TASK_TYPE,
]
started = time.perf_counter()
subprocess.run(cmd, check=True)
print('notebook elapsed minutes:', round((time.perf_counter() - started) / 60, 2))

### 6. 선택 근거와 전체 학습 결과 확인

In [ ]:
SUMMARY_PATH = OUTPUT_DIR / 'training_summary.json'
with SUMMARY_PATH.open(encoding='utf-8') as handle:
    summary = json.load(handle)
assert summary['schema_version'] == 3
assert summary['selected_route'] == 'current_r__latest_f1_f'
assert summary['validation_reference']['selection_checks']['selected_route']
display(pd.DataFrame([
    {'model': 'LightGBM', **{k: summary['lgbm'][k] for k in ['train_rows', 'target_rate', 'half_life', 'feature_count', 'train_seconds']}},
    {'model': 'CatBoost base', **{k: summary['catboost_base'][k] for k in ['train_rows', 'target_rate', 'half_life', 'feature_count', 'train_seconds']}},
    {'model': 'CatBoost F route', **{k: summary['catboost_f_route'][k] for k in ['train_rows', 'target_rate', 'half_life', 'feature_count', 'train_seconds']}},
]))
display(pd.DataFrame([summary['validation_reference']]).drop(columns=['note', 'selection_checks']))

### 7. 피처 중요도 비교

In [ ]:
lgbm_importance = pd.read_csv(OUTPUT_DIR / 'lgbm_feature_importance.csv')
base_importance = pd.read_csv(OUTPUT_DIR / 'catboost_base_feature_importance.csv')
route_importance = pd.read_csv(OUTPUT_DIR / 'catboost_f_route_feature_importance.csv')
print('LightGBM top 15')
display(lgbm_importance.head(15))
print('CatBoost base top 15')
display(base_importance.head(15))
print('CatBoost F route top 15')
display(route_importance.head(15))

### 8. 직렬화·R/F 라우팅·행 독립성·전체 test 검사

In [ ]:
audit = summary['audit']
assert audit['status'] == 'passed'
assert max(audit['native_reload_max_abs_difference'].values()) <= 1e-12
assert max(audit['routing_max_abs_difference'].values()) <= 1e-12
assert max(audit['row_independence_max_abs_difference'].values()) <= 1e-12
assert summary['end_to_end']['returncode'] == 0
display(pd.DataFrame([audit['native_reload_max_abs_difference']]))
display(pd.DataFrame([audit['routing_max_abs_difference']]))
display(pd.DataFrame([audit['row_independence_max_abs_difference']]))
display(pd.DataFrame([summary['end_to_end']]))
print('separate priors:', audit['catboost_base_target_prior'], audit['catboost_f_route_target_prior'])

### 9. 제출 ZIP 구조·파일명·해시 확인

In [ ]:
SUBMIT_ZIP = Path(summary['package']['zip_path'])
expected_members = {
    'model/lgbm_model.txt', 'model/catboost_base.cbm',
    'model/catboost_f_route.cbm', 'model/artifacts.joblib',
    'script.py', 'requirements.txt',
}
with zipfile.ZipFile(SUBMIT_ZIP) as archive:
    members = set(archive.namelist())
    corrupt = archive.testzip()
assert SUBMIT_ZIP.name == 'submit_03.zip' and len(SUBMIT_ZIP.name) <= 30
assert members == expected_members and corrupt is None
print('submission ZIP:', SUBMIT_ZIP)
print('filename chars:', len(SUBMIT_ZIP.name))
print('size (MB):', round(SUBMIT_ZIP.stat().st_size / 1e6, 3))
print('SHA-256:', summary['package']['zip_sha256'])
display(pd.DataFrame(summary['package']['members']).T)

### 10. 이번 제출의 해석

`submit_03.zip`은 submit_02의 R 예측 경로를 유지하면서 F 행만 07b의 안전한 경로로 교체한다. 07b 최근 OOF 개선은 BSS raw 약 +7.39이므로 큰 점프를 보장하는 실험은 아니다. 리더보드로 OOF 방향 일치 여부를 확인한 뒤, 다음 단계에서 독립적인 새 모델 계열을 추가한다.

In [ ]:
validation = summary['validation_reference']
base_recent_bss = routing_summary['source_audit']['source_base_recent_bss_raw']
display(pd.DataFrame([
    {'submission': 'submit_02', 'recent_bss_raw': base_recent_bss, 'role': 'current CatBoost on R/F'},
    {'submission': 'submit_03', 'recent_bss_raw': validation['recent_bss_raw'], 'role': 'base CatBoost on R, latest-F route on F'},
]))
print('리더보드 제출 시 메모: 07c current_r__latest_f1_f / no calibration')

### 11. 검토용 결과 ZIP 생성

실행 후 `submit_03.zip`, `submit03_review.zip`, 실행 결과가 저장된 07c 노트북을 전달한다.

In [ ]:
review_files = [
    OUTPUT_DIR / 'training_summary.json',
    OUTPUT_DIR / 'lgbm_feature_importance.csv',
    OUTPUT_DIR / 'catboost_base_feature_importance.csv',
    OUTPUT_DIR / 'catboost_f_route_feature_importance.csv',
]
REVIEW_ZIP = OUTPUT_DIR / 'submit03_review.zip'
with zipfile.ZipFile(REVIEW_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in review_files:
        if not path.exists():
            raise FileNotFoundError(path)
        archive.write(path, arcname=path.name)
print('review ZIP:', REVIEW_ZIP)
print('review size (MB):', round(REVIEW_ZIP.stat().st_size / 1e6, 3))
print('함께 전달:', SUBMIT_ZIP.name, REVIEW_ZIP.name, '07c_train_submit.ipynb')